In [68]:
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# The same colours the experiment scripts use, so notebook figures and run figures
# can be read side by side.
METRIC_ORDER = ["mmd_rbf03", "mmd_rbf15", "mmd_rbf75", "swd", "chamfer"]
METRIC_COLOURS = {"mmd_rbf03": "#2a78d6", "mmd_rbf15": "#eb6834", "mmd_rbf75": "#1baf7a",
                  "swd": "#eda100", "chamfer": "#4a3aa7"}
INK, MUTED, BAND = "#0b0b0b", "#52514e", "#9aa0a6"
# One colour per recording, running red to purple in spectrum order so the slowest
# wheel is at the red end and the fastest at the violet end. Yellow is left out: it
# sits where green is here, and a thin yellow line on white is unreadable.
GROUP_COLOURS = ["#d1352b", "#e07b28", "#2f9e44", "#1f6fb4", "#7b3ea1"]


def read_run(path):
    """One results.csv plus the run_config.yaml sitting beside it.

    The rotation period comes from the config rather than the results, so figures can
    name the chopper speed the recording was made at. f1 to f5 are folder names and
    mean nothing on a figure; 0.51 rev/s does.
    """
    frame = pd.read_csv(path)
    folder = Path(path).parent
    frame["run"] = folder.name
    config = yaml.safe_load((folder / "run_config.yaml").read_text(encoding="utf-8"))
    frame["rotation_period_us"] = int(config["trial"]["rotation_period_us"])
    return frame


def trial_code(name):
    """optical_chopper_data_f1 -> f1, the label the recordings are filed under."""
    return str(name).split("_")[-1]


def speed(frame):
    """The wheel's rotation frequency with its recording's code: '0.51 Hz (f1)'.

    The full rotation, not the aperture or blade-edge rate. Several recordings in one
    frame are all listed, so a figure that mixes them says so.
    """
    pairs = (frame[["rotation_period_us", "trial"]].drop_duplicates()
             .assign(hz=lambda d: 1e6 / d.rotation_period_us)
             .sort_values("hz"))
    return ", ".join(f"{row.hz:.2f} Hz ({trial_code(row.trial)})"
                     for row in pairs.itertuples())


def dataset_colour(frame):
    """The colour a recording is drawn in, matching its colour in 1.2 and 1.3.

    Keyed on position in DATASETS, so one recording keeps the same colour whether it
    is alone on its own figure or one of five in a shared panel.
    """
    order = list(globals().get("DATASETS") or sorted(frame.trial.unique()))
    name = frame.trial.iloc[0]
    index = order.index(name) if name in order else 0
    return GROUP_COLOURS[index % len(GROUP_COLOURS)]


def condition(frame):
    """One plain sentence saying what the recording condition is."""
    periods = sorted(frame.rotation_period_us.unique())
    if len(periods) > 1:
        return f"{len(periods)} recordings, rotation frequency {speed(frame)}."
    return (f"The chopper wheel completes one full rotation every "
            f"{float(periods[0]) / 1e6:.2f} s.")


def ms(microseconds):
    """5000 -> '5 ms'. Window and span lengths read better in ms than in us."""
    return f"{float(microseconds) / 1000:g} ms"

# 1. Null test

In [73]:
# Read one or more output files to generate plots.

base = "C:/Users/cxm3593/Academic/Workspace/EventSimilarityAnalysis/output/tests"

# Old data with spikes
# eval_path_lists = [
#     f"{base}/null_ruler_spikes_error/optical_chopper_data_f1/test1_null/20260824_124132/results.csv",
#     f"{base}/null_ruler_spikes_error/optical_chopper_data_f2/test1_null/20260825_064659/results.csv",
#     f"{base}/null_ruler_spikes_error/optical_chopper_data_f3/test1_null/20260825_072201/results.csv",
#     f"{base}/null_ruler_spikes_error/optical_chopper_data_f4/test1_null/20260825_080519/results.csv",
#     f"{base}/null_ruler_spikes_error/optical_chopper_data_f5/test1_null/20260825_085656/results.csv",
# ]

eval_path_lists = [
    f"{base}/optical_chopper_data_f1/test1_null/20260830_204353/results.csv",
    f"{base}/optical_chopper_data_f2/test1_null/20260830_211355/results.csv",
    f"{base}/optical_chopper_data_f3/test1_null/20260830_214903/results.csv",
    f"{base}/optical_chopper_data_f4/test1_null/20260830_223236/results.csv",
    f"{base}/optical_chopper_data_f5/test1_null/20260830_232355/results.csv",
]

nulltest_dataframe = pd.concat([read_run(path) for path in eval_path_lists],
                               ignore_index=True)

METRICS = [m for m in METRIC_ORDER if m in set(nulltest_dataframe.metric)]
WINDOWS = sorted(nulltest_dataframe.window_length_us.unique())
PERIODS = sorted(nulltest_dataframe.period_index.unique())
# Datasets in order of rotation frequency, slowest wheel first.
DATASETS = list(nulltest_dataframe.sort_values("rotation_period_us", ascending=False)
                .trial.unique())

print(f"{len(nulltest_dataframe):,} rows from {len(eval_path_lists)} run(s)")
print(f"metrics  : {', '.join(METRICS)}")
print(f"windows  : {WINDOWS} us")
print(f"segments : {PERIODS}")
print(f"seeds    : {sorted(nulltest_dataframe.split_seed.unique())}")
for name in DATASETS:
    block = nulltest_dataframe[nulltest_dataframe.trial == name]
    print(f"   {name}  {speed(block)}  {len(block):,} rows")

440,775 rows from 5 run(s)
metrics  : mmd_rbf03, mmd_rbf15, mmd_rbf75, swd, chamfer
windows  : [np.int64(1000), np.int64(5000), np.int64(10000)] us
segments : [np.int64(0), np.int64(1), np.int64(2)]
seeds    : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
   optical_chopper_data_f1  0.51 Hz (f1)  191,175 rows
   optical_chopper_data_f2  1.00 Hz (f2)  97,500 rows
   optical_chopper_data_f3  1.51 Hz (f3)  64,650 rows
   optical_chopper_data_f4  2.01 Hz (f4)  48,375 rows
   optical_chopper_data_f5  2.49 Hz (f5)  39,075 rows


## 1.1 Null reading against window start time

One figure per recording, one panel per metric. Each panel holds a single curve, so
nothing is colour-coded: the faint lines are the five random half-splits and the solid
line is their mean. The shaded bands mark the three rotation segments.

Pass `value="distance_squared"` for the unclamped quantity - `mmd.py` returns
`sqrt(max(mmd_squared, 0))`, so many MMD readings sit at exactly zero in `distance`.

In [74]:
def segment_bounds(frame, window_us):
    """(period_index, calibrated period start, calibrated period end), in us."""
    subset = frame[frame.window_length_us == window_us]
    bounds = []
    for period, block in subset.groupby("period_index"):
        start = int(block.window_start_us.min())
        period_us = int(block.rotation_period_us.iloc[0])
        bounds.append((int(period), start, start + period_us))
    return sorted(bounds)


def null_curve(frame, window_us, value="distance", metrics=None, show_seeds=True,
               height_per_metric=190):
    """Null reading against absolute window start time, one row per metric."""
    subset = frame[frame.window_length_us == window_us]
    metrics = metrics or [m for m in METRIC_ORDER if m in set(subset.metric)]
    bounds = segment_bounds(frame, window_us)

    figure = make_subplots(rows=len(metrics), cols=1, shared_xaxes=True,
                           vertical_spacing=0.03, subplot_titles=metrics)
    for row, key in enumerate(metrics, start=1):
        # One curve per panel, so colour does not separate metrics here. It marks
        # which recording this is, matching the colours used in 1.2 and 1.3.
        block = subset[subset.metric == key]
        colour = dataset_colour(subset)

        if show_seeds:
            for seed, run in block.groupby("split_seed"):
                run = run.sort_values("window_start_us")
                figure.add_trace(go.Scattergl(
                    x=run.window_start_us, y=run[value], mode="lines",
                    line=dict(color=colour, width=0.8), opacity=0.30,
                    name=f"{key} seed {seed}", showlegend=False,
                    hoverinfo="skip"), row=row, col=1)

        mean_curve = block.groupby("window_start_us")[value].mean().sort_index()
        figure.add_trace(go.Scattergl(
            x=mean_curve.index, y=mean_curve.values, mode="lines", name=key,
            line=dict(color=colour, width=1.8), showlegend=False), row=row, col=1)

        # Shade alternate segments, and label every segment on the top panel. This has
        # to come after the traces: add_vrect silently drops the shape if the subplot
        # it is aimed at is still empty.
        for period, start, end in bounds:
            if period % 2 == 0:
                figure.add_vrect(x0=start, x1=end, row=row, col=1, fillcolor=BAND,
                                 opacity=0.13, line_width=0, layer="below")
            if row == 1:
                figure.add_annotation(x=(start + end) / 2, y=1.0, yref="y domain",
                                      text=f"segment {period}", showarrow=False,
                                      yanchor="bottom", font=dict(size=11, color=MUTED),
                                      row=row, col=1)

        figure.update_yaxes(title_text=value, row=row, col=1)

    figure.update_xaxes(title_text="window start (us)", row=len(metrics), col=1)
    figure.update_layout(
        title=(f"Null test: rotation frequency {speed(subset)}, {ms(window_us)} windows"
               f"<br><sup>{condition(subset)} Faint lines are the five random "
               "half-splits, solid line is their mean. Shaded bands are the "
               f"rotation segments. y axis: {value}</sup>"),
        template="plotly_white", height=height_per_metric * len(metrics),
        margin=dict(l=80, r=30, t=90, b=60))
    return figure

In [ ]:
# for trial in DATASETS:
#     null_curve(nulltest_dataframe[nulltest_dataframe.trial == trial], 1000).show()

In [75]:
for trial in DATASETS:
    for window_us in WINDOWS:
        null_curve(nulltest_dataframe[nulltest_dataframe.trial == trial],
                   window_us).show()

## 1.2 Datasets compared

The three rotation segments are averaged into a single number, so each recording is one
point. All five sit on the same panel, ordered by how fast the wheel was turning.

The error bar is the standard deviation over everything behind that point: every
window, every random half-split, and all three segments.

In [31]:
def dataset_table(frame, window_us, value="distance"):
    """One row per (metric, dataset), with the three segments pooled together."""
    subset = frame[frame.window_length_us == window_us]
    table = (subset.groupby(["metric", "trial", "rotation_period_us"])[value]
             .agg(mean="mean", sd="std", n="count", median="median",
                  q95=lambda s: s.quantile(0.95))
             .reset_index())
    table["rotation_hz"] = 1e6 / table.rotation_period_us
    table.insert(0, "window_length_us", window_us)
    return table.sort_values(["metric", "rotation_hz"])


def dataset_points(frame, window_us, value="distance", metrics=None,
                   height_per_metric=200):
    """One point per dataset, the three segments pooled, all five on one panel."""
    table = dataset_table(frame, window_us, value)
    metrics = metrics or [m for m in METRIC_ORDER if m in set(table.metric)]
    order = list(table.sort_values("rotation_hz").trial.unique())

    figure = make_subplots(rows=len(metrics), cols=1, shared_xaxes=True,
                           vertical_spacing=0.04, subplot_titles=metrics)
    for row, key in enumerate(metrics, start=1):
        block = table[table.metric == key]
        for index, trial in enumerate(order):
            point = block[block.trial == trial]
            if point.empty:
                continue
            figure.add_trace(go.Scatter(
                x=point.rotation_hz, y=point["mean"], mode="markers",
                name=f"{point.rotation_hz.iloc[0]:.2f} Hz ({trial_code(trial)})",
                legendgroup=trial, showlegend=(row == 1),
                marker=dict(size=11,
                            color=GROUP_COLOURS[index % len(GROUP_COLOURS)]),
                error_y=dict(type="data", array=point.sd, visible=True,
                             thickness=1.4, width=8),
                customdata=np.stack([point.sd, point.n], axis=-1),
                hovertemplate=("%{x:.2f} Hz<br>mean %{y:.5g}"
                               "<br>sd %{customdata[0]:.5g}"
                               "<br>n %{customdata[1]:,}<extra></extra>")),
                row=row, col=1)
        figure.update_yaxes(title_text=value, row=row, col=1)

    figure.update_xaxes(title_text="rotation frequency (Hz)", row=len(metrics), col=1)
    figure.update_layout(
        title=(f"Null test by dataset: {ms(window_us)} windows"
               "<br><sup>each point is one recording, pooling every window, every "
               "random half-split and all three rotation segments. Bar is one "
               f"standard deviation. y axis: {value}</sup>"),
        template="plotly_white", height=height_per_metric * len(metrics),
        margin=dict(l=80, r=30, t=110, b=60),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0))
    return figure

In [32]:
dataset_points(nulltest_dataframe, 1000).show()

In [33]:
for window_us in WINDOWS:
    dataset_points(nulltest_dataframe, window_us).show()

pd.concat([dataset_table(nulltest_dataframe, w) for w in WINDOWS], ignore_index=True)

,window_length_us,metric,trial,rotation_period_us,mean,sd,n,median,q95,rotation_hz
0,1000,chamfer,optical_chopper_data_f1,1961623,11.712844,1.655517,29430,11.139260,15.425152,0.509782
1,1000,chamfer,optical_chopper_data_f2,1000183,9.168858,1.215970,15015,8.720968,11.888376,0.999817
2,1000,chamfer,optical_chopper_data_f3,664309,8.046012,1.027901,9975,7.645436,10.373318,1.505324
3,1000,chamfer,optical_chopper_data_f4,497822,7.415834,0.923673,7470,7.043015,9.483095,2.008750
4,1000,chamfer,optical_chopper_data_f5,401164,7.015893,0.863067,6030,6.666529,8.952783,2.492746
...,...,...,...,...,...,...,...,...,...,...
70,10000,swd,optical_chopper_data_f1,1961623,2.677250,0.864096,2955,2.528582,4.153040,0.509782
71,10000,swd,optical_chopper_data_f2,1000183,2.103393,1.312228,1515,1.899755,3.166471,0.999817
72,10000,swd,optical_chopper_data_f3,664309,1.663249,0.484253,1005,1.594026,2.534590,1.505324
73,10000,swd,optical_chopper_data_f4,497822,1.471037,0.433327,750,1.426511,2.226041,2.008750


## 1.3 All three window lengths, all five datasets

The same pooling as 1.2, with window length along the x axis, so each recording
contributes three points.

In [34]:
def dataset_points_all_windows(frame, value="distance", metrics=None,
                               height_per_metric=220):
    """1.2 at every window length: x is the window, colour is the dataset."""
    windows = sorted(frame.window_length_us.unique())
    table = pd.concat([dataset_table(frame, w, value) for w in windows],
                      ignore_index=True)
    metrics = metrics or [m for m in METRIC_ORDER if m in set(table.metric)]
    order = list(table.sort_values("rotation_hz").trial.unique())
    # Nudge the datasets apart so five error bars at one window length stay readable.
    offsets = np.linspace(-0.16, 0.16, len(order)) if len(order) > 1 else [0.0]
    positions = {w: i for i, w in enumerate(windows)}

    figure = make_subplots(rows=len(metrics), cols=1, shared_xaxes=True,
                           vertical_spacing=0.04, subplot_titles=metrics)
    for row, key in enumerate(metrics, start=1):
        block = table[table.metric == key]
        for index, trial in enumerate(order):
            line = block[block.trial == trial].sort_values("window_length_us")
            if line.empty:
                continue
            figure.add_trace(go.Scatter(
                x=[positions[w] + offsets[index] for w in line.window_length_us],
                y=line["mean"], mode="markers",
                name=f"{line.rotation_hz.iloc[0]:.2f} Hz ({trial_code(trial)})",
                legendgroup=trial, showlegend=(row == 1),
                marker=dict(size=9,
                            color=GROUP_COLOURS[index % len(GROUP_COLOURS)]),
                error_y=dict(type="data", array=line.sd, visible=True,
                             thickness=1.4, width=6),
                customdata=np.stack([line.window_length_us, line.sd, line.n], axis=-1),
                hovertemplate=("%{customdata[0]:,} us windows<br>mean %{y:.5g}"
                               "<br>sd %{customdata[1]:.5g}"
                               "<br>n %{customdata[2]:,}<extra></extra>")),
                row=row, col=1)
        figure.update_yaxes(title_text=value, row=row, col=1)

    figure.update_xaxes(title_text="window length", tickmode="array",
                        tickvals=list(positions.values()),
                        ticktext=[ms(w) for w in windows],
                        range=[-0.5, len(windows) - 0.5],
                        row=len(metrics), col=1)
    figure.update_layout(
        title=("Null test by dataset: every window length"
               "<br><sup>each point is one recording at one window length, pooling "
               "every window, every random half-split and all three rotation "
               f"segments. Bar is one standard deviation. y axis: {value}</sup>"),
        template="plotly_white", height=height_per_metric * len(metrics),
        margin=dict(l=80, r=30, t=120, b=60),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0))
    return figure

In [35]:
dataset_points_all_windows(nulltest_dataframe).show()

# 2. Ruler test

A reference span compared against the span starting `shift` later in the same
recording, sweeping `shift`. The span is one whole rotation, cut into windows; window i
of the reference is compared against window i of the shifted span and the result is
averaged over every window in the span.

In [36]:
# Read one or more output files to generate plots.

ruler_path_lists = [
    f"{base}/optical_chopper_data_f1/test2_ruler/20260824_192020_period_w5000us/results.csv",
    f"{base}/optical_chopper_data_f2/test2_ruler/20260824_220316_period_w5000us/results.csv",
    f"{base}/optical_chopper_data_f3/test2_ruler/20260825_001934_period_w5000us/results.csv",
    f"{base}/optical_chopper_data_f4/test2_ruler/20260825_023131_period_w5000us/results.csv",
    f"{base}/optical_chopper_data_f5/test2_ruler/20260825_043745_period_w5000us/results.csv",
]

ruler_dataframe = pd.concat([read_run(path) for path in ruler_path_lists],
                            ignore_index=True)

RULER_WINDOWS = sorted(ruler_dataframe.window_length_us.unique())
RULER_DATASETS = list(ruler_dataframe.sort_values("rotation_period_us", ascending=False)
                      .trial.unique())

print(f"{len(ruler_dataframe):,} rows from {len(ruler_path_lists)} run(s)")
print(f"windows : {RULER_WINDOWS} us")
for name in RULER_DATASETS:
    block = ruler_dataframe[ruler_dataframe.trial == name]
    print(f"   {name}  {speed(block)}  {block.shift_us.nunique()} shifts, "
          f"0 to {block.shift_us.max():,} us, "
          f"{int(block.n_windows.iloc[0]):,} window pairs per point")

1,325 rows from 5 run(s)
windows : [np.int64(5000)] us
   optical_chopper_data_f1  0.51 Hz (f1)  106 shifts, 0 to 1,960,000 us, 393 window pairs per point
   optical_chopper_data_f2  1.00 Hz (f2)  58 shifts, 0 to 1,000,000 us, 201 window pairs per point
   optical_chopper_data_f3  1.51 Hz (f3)  41 shifts, 0 to 660,000 us, 133 window pairs per point
   optical_chopper_data_f4  2.01 Hz (f4)  32 shifts, 0 to 480,000 us, 100 window pairs per point
   optical_chopper_data_f5  2.49 Hz (f5)  28 shifts, 0 to 400,000 us, 81 window pairs per point


## 2.1 Distance against shift

One figure per recording, drawn in that recording's colour. One panel per metric.

Each point is the whole rotation compared against itself shifted by that much, averaged
over every window pair inside the rotation. `shift = 0` is the rotation against itself,
which is exactly zero for every metric.

The shaded strip on the left is where the shift is smaller than the window length, so
the two windows still overlap in time and share events.

The error bar is `mean_sd_within_span`: how much the metric varies across the window
pairs inside one rotation. It is not `sd_distance`, which is the spread across reference
positions and is zero here because each run used a single reference position.

In [37]:
def ruler_curve(frame, window_us=None, value="mean_distance", spread="mean_sd_within_span",
                x_max=None, metrics=None, shade_overlap=True, height_per_metric=190):
    """Ruler curve for one recording, one panel per metric."""
    subset = frame if window_us is None else frame[frame.window_length_us == window_us]
    window_us = window_us or int(subset.window_length_us.iloc[0])
    metrics = metrics or [m for m in METRIC_ORDER if m in set(subset.metric)]
    colour = dataset_colour(subset)

    if x_max is not None:
        subset = subset[subset.shift_us <= x_max]

    figure = make_subplots(rows=len(metrics), cols=1, shared_xaxes=True,
                           vertical_spacing=0.035, subplot_titles=metrics)
    for row, key in enumerate(metrics, start=1):
        block = subset[subset.metric == key].sort_values("shift_us")
        bars = (dict(type="data", array=block[spread], visible=True,
                     thickness=0.8, width=0)
                if spread and spread in block.columns else None)
        figure.add_trace(go.Scatter(
            x=block.shift_us, y=block[value], mode="lines+markers",
            name=key, showlegend=False,
            line=dict(color=colour, width=2), marker=dict(size=4),
            error_y=bars), row=row, col=1)

        # After the traces: add_vrect drops the shape if the panel is still empty.
        if shade_overlap and window_us > 0:
            figure.add_vrect(x0=0, x1=window_us, row=row, col=1, fillcolor=BAND,
                             opacity=0.16, line_width=0, layer="below")
            if row == 1:
                figure.add_annotation(x=window_us / 2, y=1.0, yref="y domain",
                                      text="windows overlap", showarrow=False,
                                      yanchor="bottom", font=dict(size=11, color=MUTED),
                                      row=row, col=1)
        figure.update_yaxes(title_text=value, row=row, col=1)

    figure.update_xaxes(title_text="shift (us)", row=len(metrics), col=1)
    pairs = (f", averaged over {int(subset.n_windows.iloc[0])} window pairs"
             if "n_windows" in subset.columns else "")
    figure.update_layout(
        title=(f"Ruler test: rotation frequency {speed(subset)}"
               f"<br><sup>{condition(subset)} One full rotation shifted against itself "
               f"and compared in {ms(window_us)} windows{pairs}. Bar is the spread "
               f"across those pairs. y axis: {value}</sup>"),
        template="plotly_white", height=height_per_metric * len(metrics),
        margin=dict(l=80, r=30, t=110, b=60))
    return figure

In [38]:
for trial in RULER_DATASETS:
    ruler_curve(ruler_dataframe[ruler_dataframe.trial == trial]).show()

## 2.2 All five recordings on one map

Every ruler curve on shared axes, coloured by recording. The x axis runs to the longest
sweep, which is f1's - a full rotation there is 1.96 s against 0.40 s at f5, so the
faster wheels finish early and their curves stop partway across. Each metric panel
shares one y range across all five.

Two things to keep in mind reading it. The faster recordings have fewer points, because
every sweep steps in 20 ms and their rotations are shorter: 106 points at f1 down to 28
at f5. And the y values are not strictly comparable between recordings - a 5 ms window
holds about 7,300 events at f1 and 30,000 at f5, and MMD in particular moves with
sample size.

In [39]:
def ruler_overlay(frame, value="mean_distance", metrics=None, spread=None,
                  x_as_fraction=False, height_per_metric=200):
    """Every recording's ruler on one set of axes, one panel per metric.

    x_as_fraction divides the shift by each recording's own rotation, which lines the
    curves up on the turn of the wheel rather than on absolute time.
    """
    metrics = metrics or [m for m in METRIC_ORDER if m in set(frame.metric)]
    order = list(frame.sort_values("rotation_period_us", ascending=False).trial.unique())

    figure = make_subplots(rows=len(metrics), cols=1, shared_xaxes=True,
                           vertical_spacing=0.035, subplot_titles=metrics)
    for row, key in enumerate(metrics, start=1):
        for index, trial in enumerate(order):
            block = (frame[(frame.metric == key) & (frame.trial == trial)]
                     .sort_values("shift_us"))
            if block.empty:
                continue
            x = (block.shift_us / block.rotation_period_us if x_as_fraction
                 else block.shift_us)
            bars = (dict(type="data", array=block[spread], visible=True,
                         thickness=0.8, width=0)
                    if spread and spread in block.columns else None)
            figure.add_trace(go.Scatter(
                x=x, y=block[value], mode="lines+markers",
                name=speed(block), legendgroup=trial, showlegend=(row == 1),
                line=dict(color=GROUP_COLOURS[index % len(GROUP_COLOURS)], width=1.8),
                marker=dict(size=4), error_y=bars,
                hovertemplate="%{x:,.4g}<br>%{y:.5g}<extra>" + speed(block) + "</extra>"),
                row=row, col=1)
        figure.update_yaxes(title_text=value, row=row, col=1)

    figure.update_xaxes(
        title_text=("shift (fraction of a rotation)" if x_as_fraction
                    else "shift (us), running to the longest rotation"),
        row=len(metrics), col=1)
    figure.update_layout(
        title=("Ruler test: all five recordings"
               "<br><sup>one full rotation shifted against itself, compared in 5 ms "
               "windows and averaged over every window pair in the rotation. "
               f"y axis: {value}</sup>"),
        template="plotly_white", height=height_per_metric * len(metrics),
        margin=dict(l=80, r=30, t=110, b=60),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0))
    return figure

In [40]:
ruler_overlay(ruler_dataframe).show()

<!-- codex-test34-analysis-v1 -->
# 3. Modifier test

These figures use the existing **f1, 1 ms preliminary runs** to build and validate the
analysis. They are not the final benchmark results. The configuration table below makes
the sampling choices visible: the current runs use 40 phase-spread windows per rotation
and at most 750 events on each side of a comparison.

Test 3 asks two questions:

1. **Generic selectivity:** which controlled changes does each metric detect?
2. **Directed hypotheses:** does making real data more like v2e along a measured defect
   reduce the real–v2e distance?

In [41]:
from IPython.display import Markdown, display


TEST_ROOT = Path(
    "C:/Users/cxm3593/Academic/Workspace/EventSimilarityAnalysis/output/tests"
)
TEST3A_RUN = (TEST_ROOT / "optical_chopper_data_f1" / "test3_modifier"
              / "20260824_022319_3a_w1000us")
TEST3B_RUN = (TEST_ROOT / "optical_chopper_data_f1" / "test3_modifier"
              / "20260824_022810_3b_w1000us")


def read_bundle(folder):
    # Read every standard output that exists beside one experiment run.
    folder = Path(folder)
    bundle = {
        "folder": folder,
        "config": yaml.safe_load((folder / "run_config.yaml").read_text(encoding="utf-8")),
    }
    for name in ("results", "summary", "placement"):
        path = folder / f"{name}.csv"
        bundle[name] = pd.read_csv(path) if path.exists() else None
    return bundle


def run_settings(bundle, label):
    # One visible row of the choices that materially affect interpretation.
    parameters = bundle["config"]["parameters"]
    periods = parameters.get("periods")
    if periods is None:
        periods = parameters.get("n_periods_used")
    return {
        "run": label,
        "trial": bundle["config"]["trial"]["name"],
        "phase": parameters.get("phase", "test4"),
        "window_us": parameters.get("window_us"),
        "periods": periods,
        "max_windows": parameters.get("max_windows"),
        "events_per_comparison": parameters.get("events_per_comparison"),
        "seeds": parameters.get("seeds", parameters.get("seed")),
        "metrics": ", ".join(parameters.get("metrics", [])),
        "result_rows": len(bundle["results"]) if bundle.get("results") is not None else 0,
    }


test3a = read_bundle(TEST3A_RUN)
test3b = read_bundle(TEST3B_RUN)
display(Markdown(
    "> **Preliminary pipeline-validation data.** These settings are displayed so the "
    "meeting figures cannot be mistaken for the final run."
))
display(pd.DataFrame([
    run_settings(test3a, "Test 3a — generic"),
    run_settings(test3b, "Test 3b — directed"),
]))

> **Preliminary pipeline-validation data.** These settings are displayed so the meeting figures cannot be mistaken for the final run.

,run,trial,phase,window_us,periods,max_windows,events_per_comparison,seeds,metrics,result_rows
0,Test 3a — generic,optical_chopper_data_f1,3a,1000,3,40,750,"[0, 1, 2]","mmd_rbf03, mmd_rbf15, mmd_rbf75, swd, chamfer",66600
1,Test 3b — directed,optical_chopper_data_f1,3b,1000,3,40,750,"[0, 1, 2]","mmd_rbf03, mmd_rbf15, mmd_rbf75, swd, chamfer",18000


## 3.1 Generic modifier sweeps

Each cell in the selectivity grid is one metric responding to one controlled modifier.
The axes retain the metric's native units; values are not normalised across metrics.
The useful questions are whether the response is monotonic and whether different metrics
respond to different modifier types.

In [42]:
MODIFIER_ORDER = [
    "spatial_offset_x", "spatial_offset_xy", "scaling", "subsample",
    "uniform_noise", "temporal_clump_uniform",
]
MODIFIER_LABELS = {
    "spatial_offset_x": "spatial offset x (px)",
    "spatial_offset_xy": "spatial offset x+y (px)",
    "scaling": "spatial scale",
    "subsample": "subsampling (retained fraction)",
    "uniform_noise": "uniform noise (added fraction)",
    "temporal_clump_uniform": "temporal clump (us)",
}
METRIC_LABELS = {
    "mmd_rbf03": "MMD RBF-3", "mmd_rbf15": "MMD RBF-15",
    "mmd_rbf75": "MMD RBF-75", "swd": "SWD", "chamfer": "Chamfer",
}


def selectivity_grid(summary, modifiers=None, metrics=None):
    # Raw response curves, one modifier by metric cell.
    modifiers = modifiers or [m for m in MODIFIER_ORDER if m in set(summary.modifier)]
    metrics = metrics or [m for m in METRIC_ORDER if m in set(summary.metric)]
    figure = make_subplots(
        rows=len(modifiers), cols=len(metrics),
        horizontal_spacing=0.035, vertical_spacing=0.055,
        subplot_titles=[METRIC_LABELS.get(metric, metric) if row == 0 else ""
                        for row in range(len(modifiers)) for metric in metrics],
    )
    for row, modifier in enumerate(modifiers, start=1):
        for col, metric in enumerate(metrics, start=1):
            block = summary[(summary.modifier == modifier) & (summary.metric == metric)]
            block = block.sort_values("magnitude")
            figure.add_trace(go.Scatter(
                x=block.magnitude, y=block.mean_distance,
                mode="lines+markers", showlegend=False,
                line=dict(color=METRIC_COLOURS[metric], width=2),
                marker=dict(size=5),
                error_y=dict(type="data", array=block.sd_distance,
                             thickness=0.8, width=0, visible=True),
                customdata=np.stack([block.sd_distance, block.n_comparisons], axis=-1),
                hovertemplate=("magnitude %{x:g}<br>mean %{y:.5g}"
                               "<br>sd %{customdata[0]:.4g}"
                               "<br>n %{customdata[1]:,.0f}<extra></extra>"),
            ), row=row, col=col)
            if col == 1:
                figure.update_yaxes(title_text=MODIFIER_LABELS[modifier], row=row, col=col)
    figure.update_layout(
        title=("Test 3a — metric selectivity under controlled modifications"
               "<br><sup>preliminary f1 result, 1 ms windows; raw metric units</sup>"),
        template="plotly_white", height=235 * len(modifiers), width=1350,
        margin=dict(l=155, r=25, t=100, b=50),
    )
    return figure


selectivity_grid(test3a["summary"]).show()

## 3.2 Subsampling as a negative control

This is the same **subsampling** modifier shown in the selectivity grid. The x axis is
the fraction of events retained: `1.0` keeps every event, `0.5` keeps half, and `0.25`
keeps one quarter. Events are selected randomly without replacement.

Subsampling is a negative control because it changes the size and density of the
empirical point cloud without intentionally changing the population that generated it.
A strong response therefore reveals finite-sample or point-density sensitivity.

In [43]:
def modifier_panels(summary, modifier):
    metrics = [m for m in METRIC_ORDER if m in set(summary.metric)]
    figure = make_subplots(rows=len(metrics), cols=1, shared_xaxes=True,
                           vertical_spacing=0.04,
                           subplot_titles=[METRIC_LABELS.get(m, m) for m in metrics])
    for row, metric in enumerate(metrics, start=1):
        block = summary[(summary.modifier == modifier) & (summary.metric == metric)]
        block = block.sort_values("magnitude")
        figure.add_trace(go.Scatter(
            x=block.magnitude, y=block.mean_distance, mode="lines+markers",
            showlegend=False, line=dict(color=METRIC_COLOURS[metric], width=2),
            marker=dict(size=6),
            error_y=dict(type="data", array=block.sd_distance,
                         thickness=0.8, width=0, visible=True),
        ), row=row, col=1)
        figure.update_yaxes(title_text="distance", row=row, col=1)
    figure.update_xaxes(title_text="subsampling (fraction of events retained)",
                        range=[0.2, 1.05], row=len(metrics), col=1)
    figure.update_layout(
        title=("Test 3a — subsampling negative control"
               "<br><sup>preliminary f1 result, 1 ms windows</sup>"),
        template="plotly_white", height=175 * len(metrics), width=760,
        margin=dict(l=80, r=30, t=90, b=60),
    )
    return figure


modifier_panels(test3a["summary"], "subsample").show()

## 3.3 Directed hypotheses

This preliminary Test 3b run uses **1 ms windows**. Each displayed value averages
40 phase-spread windows × 3 rotations × 3 random seeds = 360 comparisons per metric.
The plot below uses the **natural-count** results so that event-count matching actually
changes the data.

- **Unmodified real vs v2e:** the original comparison; this is the reference value.
- **Clump real to v2e grid:** snap every real timestamp to the nearest timestamp that
  v2e actually emitted. Spatial positions and event count are unchanged.
- **Subsample v2e to real count:** randomly remove v2e events until its count equals
  the real count. This tests whether v2e's larger event count explains the gap.
- **Add uniform events to real:** add events until real has the v2e count. Added `x` is
  uniform over 0–1280, `y` over 0–720, and time over the complete 1 ms window.
- **Clump + match:** clump real timestamps and subsample v2e to the real count.

A directed hypothesis is supported only when its point falls clearly below the
unmodified reference.

In [44]:
DIRECTED_ORDER = [
    "none", "clump_to_v2e_grid", "match_count_v2e_down",
    "match_count_real_up", "clump_and_match",
]
DIRECTED_LABELS = {
    "none": "unmodified",
    "clump_to_v2e_grid": "clump real to v2e grid",
    "match_count_v2e_down": "subsample v2e to real count",
    "match_count_real_up": "add uniform events to real",
    "clump_and_match": "clump + match",
}


def directed_points(summary, sizing="natural"):
    subset = summary[summary.sizing == sizing]
    metrics = [m for m in METRIC_ORDER if m in set(subset.metric)]
    figure = make_subplots(rows=len(metrics), cols=1, shared_xaxes=True,
                           vertical_spacing=0.04,
                           subplot_titles=[METRIC_LABELS.get(m, m) for m in metrics])
    for row, metric in enumerate(metrics, start=1):
        block = (subset[subset.metric == metric].set_index("modifier")
                 .reindex(DIRECTED_ORDER).reset_index())
        colours = ["#555555" if name == "none" else METRIC_COLOURS[metric]
                   for name in block.modifier]
        figure.add_trace(go.Scatter(
            x=[DIRECTED_LABELS[name] for name in block.modifier],
            y=block.mean_distance, mode="markers", showlegend=False,
            marker=dict(size=10, color=colours),
            error_y=dict(type="data", array=block.sd_distance,
                         thickness=1, width=5, visible=True),
        ), row=row, col=1)
        figure.update_yaxes(title_text="distance", row=row, col=1)
    figure.update_layout(
        title=("Test 3b — directed hypotheses"
               "<br><sup>natural event counts; 1 ms windows; lower than unmodified supports the hypothesis</sup>"),
        template="plotly_white", height=185 * len(metrics), width=780,
        margin=dict(l=80, r=30, t=95, b=130),
    )
    figure.update_xaxes(tickangle=-22, row=len(metrics), col=1)
    return figure


def directed_change_table(summary, sizing="natural"):
    subset = summary[summary.sizing == sizing]
    pivot = subset.pivot(index="modifier", columns="metric", values="mean_distance")
    change = (pivot.divide(pivot.loc["none"], axis=1) - 1.0) * 100.0
    return change.reindex(DIRECTED_ORDER).reindex(columns=METRIC_ORDER).round(1)


directed_points(test3b["summary"], sizing="natural").show()
display(Markdown("**Percent change from the unmodified real–v2e distance:**"))
display(directed_change_table(test3b["summary"], sizing="natural"))

**Percent change from the unmodified real–v2e distance:**

metric,mmd_rbf03,mmd_rbf15,mmd_rbf75,swd,chamfer
modifier,,,,,
none,0.0,0.0,0.0,0.0,0.0
clump_to_v2e_grid,-2.2,-0.2,0.0,-0.0,-3.3
match_count_v2e_down,-0.0,-0.0,-0.4,1.8,8.8
match_count_real_up,1.6,42.7,204.9,882.2,976.7
clump_and_match,-2.2,-0.4,-0.7,2.3,5.8


# 4. Synthetic-data test

The current output compares real and v2e rotations, together with representative modified
real sources. Test 4 asks where the simulator lies relative to natural real variation and
the calibrated changes from Tests 2 and 3.

For the group meeting, use this section to demonstrate the analysis workflow. Do not
present the current f1 values as the final simulator ranking.

In [45]:
TEST4_RUN = (TEST_ROOT / "optical_chopper_data_f1" / "test4_synthetic"
             / "20260824_023419_w1000us")
test4 = read_bundle(TEST4_RUN)
display(Markdown(
    "> **Preliminary pipeline-validation data.** The final run will use the explicitly "
    "selected sampling protocol and additional trials/simulators."
))
display(pd.DataFrame([run_settings(test4, "Test 4 — real vs v2e")]))

> **Preliminary pipeline-validation data.** The final run will use the explicitly selected sampling protocol and additional trials/simulators.

,run,trial,phase,window_us,periods,max_windows,events_per_comparison,seeds,metrics,result_rows
0,Test 4 — real vs v2e,optical_chopper_data_f1,test4,1000,7,40,750,0,"mmd_rbf03, mmd_rbf15, mmd_rbf75, swd, chamfer",9800


## 4.1 Real versus real and real versus v2e

For this first view, only the two essential comparisons are shown:

- **Real:** baseline real period 0 compared with corresponding phase windows from all
  seven real periods.
- **v2e:** the same baseline real windows compared with v2e at corresponding phases.

Each point is the mean of the seven period-level means; the error bar is their spread.
The real value is not expected to be zero because it includes comparisons with other
rotations, natural rotation-to-rotation variation, phase wander, and independent
750-event subsamples. Even period 0 uses two independently selected samples for SWD and
Chamfer. Modified-real sources can be added later as a second, more detailed figure.

In [46]:
SOURCE_LABELS = {"real": "real–real", "v2e": "real–v2e"}


def source_comparison(summary, sources=("real", "v2e")):
    subset = summary[(summary.polarity_channel == "all") & summary.source.isin(sources)]
    table = (subset.groupby(["source", "metric"])
             .mean_distance.agg(["mean", "std"]).reset_index())
    metrics = [m for m in METRIC_ORDER if m in set(table.metric)]
    figure = make_subplots(rows=len(metrics), cols=1, shared_xaxes=True,
                           vertical_spacing=0.04,
                           subplot_titles=[METRIC_LABELS.get(m, m) for m in metrics])
    for row, metric in enumerate(metrics, start=1):
        block = table[table.metric == metric].set_index("source").reindex(sources).reset_index()
        colours = ["#555555" if source == "real" else "#d1352b"
                   for source in block.source]
        figure.add_trace(go.Scatter(
            x=[SOURCE_LABELS[source] for source in block.source],
            y=block["mean"], mode="markers", showlegend=False,
            marker=dict(size=12, color=colours),
            error_y=dict(type="data", array=block["std"],
                         thickness=1, width=7, visible=True),
        ), row=row, col=1)
        figure.update_yaxes(title_text="distance", row=row, col=1)
    figure.update_layout(
        title=("Test 4 — natural real variation and the real–v2e gap"
               "<br><sup>preliminary f1 result, 1 ms windows; spread is across seven periods</sup>"),
        template="plotly_white", height=175 * len(metrics), width=620,
        margin=dict(l=80, r=30, t=95, b=60),
    )
    return figure


source_comparison(test4["summary"]).show()

## 4.2 Where within the period does the gap occur?

The phase profile averages corresponding window locations across rotations. It shows
whether the real–v2e discrepancy is uniform or concentrated at particular phases of the
motion. Only real and v2e are drawn so the presentation figure stays readable.

The current profile uses 1 ms windows at 40 evenly spread phase locations in each rotation.

In [47]:
def phase_profile(results, rotation_period_us, window_us, sources=("real", "v2e")):
    subset = results[(results.polarity_channel == "all") & results.source.isin(sources)].copy()
    subset["phase_fraction"] = subset.window_index * window_us / rotation_period_us
    table = (subset.groupby(["source", "metric", "window_index", "phase_fraction"])
             .distance.agg(["mean", "std"]).reset_index())
    metrics = [m for m in METRIC_ORDER if m in set(table.metric)]
    colours = {"real": "#555555", "v2e": "#d1352b"}
    figure = make_subplots(rows=len(metrics), cols=1, shared_xaxes=True,
                           vertical_spacing=0.04,
                           subplot_titles=[METRIC_LABELS.get(m, m) for m in metrics])
    for row, metric in enumerate(metrics, start=1):
        for source in sources:
            block = table[(table.metric == metric) & (table.source == source)]
            figure.add_trace(go.Scatter(
                x=block.phase_fraction, y=block["mean"], mode="lines+markers",
                name=source, legendgroup=source, showlegend=(row == 1),
                line=dict(color=colours[source], width=2), marker=dict(size=4),
                error_y=dict(type="data", array=block["std"],
                             thickness=0.6, width=0, visible=True),
            ), row=row, col=1)
        figure.update_yaxes(title_text="distance", row=row, col=1)
    figure.update_xaxes(title_text="position within rotation", tickformat=".0%",
                        row=len(metrics), col=1)
    figure.update_layout(
        title=("Test 4 — phase profile of real and v2e"
               "<br><sup>preliminary f1 result, averaged across rotations</sup>"),
        template="plotly_white", height=175 * len(metrics), width=780,
        margin=dict(l=80, r=30, t=95, b=60),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    )
    return figure


test4_parameters = test4["config"]["parameters"]
phase_profile(
    test4["results"],
    rotation_period_us=test4["config"]["trial"]["rotation_period_us"],
    window_us=test4_parameters["window_us"],
).show()

## 4.3 Placement on the temporal ruler

The v2e distance is inverted through the calibrated temporal ruler and reported in
milliseconds. Metrics are always ordered as MMD RBF-3, MMD RBF-15, MMD RBF-75, SWD,
and Chamfer. RBF-3 is intentionally retained in the figure: its v2e value is outside
the measured ruler range, so no numerical shift is extrapolated.

In [48]:
def temporal_placement(placement):
    table = placement.copy()
    table["equivalent_shift_ms"] = table.equivalent_shift_us / 1000.0
    table["status"] = np.where(table.equivalent_shift_ms.notna(),
                               "inside calibrated range", "outside calibrated range")
    return table[["metric", "v2e_mean_distance", "floor", "ceiling",
                  "equivalent_shift_ms", "status"]].set_index("metric").reindex(METRIC_ORDER)


placement_table = temporal_placement(test4["placement"])
display(placement_table.round(4))

plot_table = placement_table.reset_index()
labels = [METRIC_LABELS.get(metric, metric) for metric in plot_table.metric]
values = plot_table.equivalent_shift_ms.fillna(0.0)
symbols = ["circle" if pd.notna(value) else "x"
           for value in plot_table.equivalent_shift_ms]
texts = [f"{value:.1f} ms" if pd.notna(value) else "outside calibrated range"
         for value in plot_table.equivalent_shift_ms]
figure = go.Figure(go.Scatter(
    x=values, y=labels, mode="markers+text",
    marker=dict(size=14, symbol=symbols,
                color=[METRIC_COLOURS[metric] for metric in plot_table.metric]),
    text=texts, textposition="middle right",
    hovertemplate="%{y}<br>%{text}<extra></extra>",
))
largest = max(1.0, float(values.max()))
figure.update_yaxes(categoryorder="array", categoryarray=list(reversed(labels)))
figure.update_xaxes(title_text="equivalent temporal shift (ms)",
                    range=[-0.03 * largest, 1.35 * largest])
figure.update_layout(
    title=("Test 4 — v2e distance expressed on the temporal ruler"
           "<br><sup>preliminary f1 result; an x marks a value outside the calibrated range</sup>"),
    yaxis_title="", template="plotly_white", height=390, width=720,
    margin=dict(l=130, r=150, t=95, b=60),
)
figure.show()

,v2e_mean_distance,floor,ceiling,equivalent_shift_ms,status
metric,,,,,
mmd_rbf03,0.0688,0.0002,0.0503,NaN,outside calibrated range
mmd_rbf15,0.1470,0.0000,0.2751,22.2238,inside calibrated range
mmd_rbf75,0.1077,0.0000,0.4887,27.0526,inside calibrated range
swd,18.2082,8.3624,47.9141,34.2469,inside calibrated range
chamfer,18.0353,11.3761,50.9827,21.1601,inside calibrated range


## 4.4 All-to-all comparison — optional diagnostic

This test compares every real rotation and every v2e rotation with each other. It is
reduced to three quantities:

- **Real–real:** natural variation among real rotations.
- **v2e–v2e:** variation among simulated rotations; a large value means the simulator
  is not self-consistent under this metric.
- **Real–v2e:** separation between the two sources.

The final column asks the simplest question: is the real–v2e distance larger than both
within-source distances? This is an optional diagnostic, not the main Test 4 result.
MDS is omitted here because it does not add a clearer quantitative conclusion.

In [49]:
def pairwise_group_summary(folder, metrics=None):
    metrics = metrics or METRIC_ORDER
    rows = []
    for metric in metrics:
        path = Path(folder) / f"all_to_all_{metric}.csv"
        if not path.exists():
            continue
        matrix = pd.read_csv(path, index_col=0)
        real = [name for name in matrix.index if str(name).startswith("real_")]
        v2e = [name for name in matrix.index if str(name).startswith("v2e_")]
        real_real = matrix.loc[real, real].to_numpy(dtype=float)
        v2e_v2e = matrix.loc[v2e, v2e].to_numpy(dtype=float)
        real_v2e = matrix.loc[real, v2e].to_numpy(dtype=float)
        rows.append({
            "metric": metric,
            "real_real": real_real[np.triu_indices(len(real), 1)].mean(),
            "v2e_v2e": v2e_v2e[np.triu_indices(len(v2e), 1)].mean(),
            "real_v2e": real_v2e.mean(),
        })
    table = pd.DataFrame(rows).set_index("metric").reindex(METRIC_ORDER)
    table["real_v2e_exceeds_both_within"] = (
        table.real_v2e > table[["real_real", "v2e_v2e"]].max(axis=1)
    ).map({True: "yes", False: "no"})
    return table


pairwise_summary = pairwise_group_summary(TEST4_RUN)
display(pairwise_summary.round(5))

,real_real,v2e_v2e,real_v2e,real_v2e_exceeds_both_within
metric,,,,
mmd_rbf03,0.01725,0.08358,0.06416,no
mmd_rbf15,0.04839,0.12398,0.11728,no
mmd_rbf75,0.04820,0.06863,0.08878,yes
swd,11.73802,12.70300,16.27037,yes
chamfer,12.46083,16.84657,16.28396,no


## Test 3–4 presentation checklist

For the group meeting, the smallest coherent set is:

1. Test 3 selectivity grid.
2. Test 3 subsampling negative control.
3. Test 3 directed-hypothesis bars.
4. Test 4 real / modified-real / v2e comparison.
5. Test 4 temporal placement in milliseconds.

The phase profile and all-to-all summary answer useful secondary questions and can be
shown only if discussion time allows. Every current figure should retain the preliminary
sampling disclaimer until the final run protocol is selected.

In [50]:
# V2CE comparison
